# Build and benchmark a research environment

This offline walkthrough uses the same `LibraryEnv` as `env.py` and `run.py`. It demonstrates:

1. tools and hidden evaluator labels,
2. framework-owned `step()` and nested tool tracing,
3. terminal rewards and discounted returns,
4. versioned benchmark tasks, and
5. comparing fresh policy instances with `Benchmark.from_policies()`.

The pattern maps directly to support agents, retrieval workflows, and other search-then-act tasks.

In [ ]:
from importlib import import_module
from pathlib import Path

from enroute import Benchmark, JSONLSink, ScriptedPolicy, TaskDataset, TraceWriter
from enroute.tracing import ParsedAction
from examples.environment.library.env import make_env

benchmark_policies = import_module("examples.benchmarking.02_compare_policies")
ResearchPolicy = benchmark_policies.ResearchPolicy
ImmediateGuessPolicy = benchmark_policies.ImmediateGuessPolicy

env = make_env()
print(f"{env.name}@{env.version}  fingerprint={env.fingerprint()[:12]}…")
print("tools:", [tool.function.name for tool in env.tool_defs])
print("tasks:", [task.task_id for task in env.iter_tasks()])

## 1. Tools, state, and hidden labels

`LibraryEnv` is tool-only, so it inherits `apply_action()`. Framework-owned `step()` executes the selected tool and records the turn.

The task's `expected` answer is available to the scorer but excluded from observations, snapshots, and trace metadata. This prevents evaluator labels from leaking to the policy.

In [ ]:
task = next(env.iter_tasks())
observation, _ = env.reset(task, model="manual-policy")
result = env.step(ParsedAction(name="research", arguments={"query": "voting"}))
manual_rollout = env.close_episode()

decision = manual_rollout.trace.decisions()[0]
print(result.observation)
print("root tool:", decision.tool_calls[0].name)
print("nested tools:", [child.name for child in decision.tool_calls[0].children])
print("expected leaked into trace:", "expected" in manual_rollout.trace.metadata["task"])

## 2. Generate training traces

A complete research episode gets its reward only after `answer`. `Trace.returns(gamma=...)` assigns discounted credit back to earlier search and read decisions. This is the key handoff from an evaluation environment to imitation learning, reward modeling, or offline RL.

In [ ]:
researcher = ScriptedPolicy(
    [
        ParsedAction(name="search", arguments={"query": "voting"}),
        ParsedAction(name="read", arguments={"doc_id": "d1"}),
        ParsedAction(name="answer", arguments={"text": "the river path"}),
    ]
)
rollout = make_env().run_episode(task, researcher, model="scripted-researcher")

print("actions:", [d.parsed_action[0].name for d in rollout.trace.decisions()])
print("terminal reward:", rollout.trace.outcome.reward)
print("discounted returns:", rollout.trace.returns(gamma=0.9))

## 3. Benchmark both default tasks

`TaskDataset` gives the suite a version and content hash. `Benchmark.from_policies()` evaluates matching `(task_id, repeat)` slots, aggregates rewards, and computes paired win rates. Each policy factory and `environment_factory` returns a fresh object for concurrency safety.

In [ ]:
template = make_env()
dataset = TaskDataset(
    name="library-notebook",
    version=template.version,
    tasks=list(template.iter_tasks()),
)

out = Path(".enroute/examples")
out.mkdir(parents=True, exist_ok=True)
writer = TraceWriter(JSONLSink(out / "library-notebook-episodes.jsonl"))
try:
    report = Benchmark.from_policies(
        template,
        {"researcher": ResearchPolicy, "guesser": ImmediateGuessPolicy},
        concurrency=2,
        environment_factory=make_env,
        trace_writer=writer,
    ).run(dataset=dataset)
finally:
    writer.close()

print(report.to_markdown())

## Adapt this pattern to your task

1. Put private workflow state in a typed `State`; expose only allowed information from `Observation`.
2. Load each case in `setup(task)`. Keep answer keys and rubrics in `TaskData.expected`, not policy-visible state.
3. Add atomic or hierarchical `@tool` actions. Tool-only environments inherit `apply_action()`; custom scalar/text environments override it and return `ActionResult`.
4. Define natural completion in `done()`, terminal quality in scorers, and optional dense feedback in `step_reward()`.
5. Version tasks with `TaskDataset`, return fresh policies/environments from factories, and retain the benchmark manifest with reports.
6. Inspect `Trace.transitions()` or `Trace.returns()` to prepare training examples.
7. For real model IDs, use `Benchmark(env, models=[...], client=Enroute()).run(dataset=dataset)` instead of `from_policies()`.

See `env.py`, `run.py`, and `examples/benchmarking/02_compare_policies.py` for the complete runnable versions.